# 05 - Hyperparameter Tuning with Optuna

Optuna tries different parameter values and keeps the one with the lowest error on a validation set taken from the training data. The test set is not touched here.

Input: train files from 04 → Output: tuned models in `models/`

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sns.set(style="darkgrid")

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
import optuna
import joblib

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [2]:
X_train = pd.read_csv('data/processed/X_train.csv')
y_train = pd.read_csv('data/processed/y_train.csv')['spend_next_month']
scaler = joblib.load('models/scaler.pkl')

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
X_tr_sc = scaler.transform(X_tr)
X_val_sc = scaler.transform(X_val)

### Tuning Random Forest

In [3]:
def rf_objective(trial):
    model = RandomForestRegressor(
        n_estimators=trial.suggest_int('n_estimators', 100, 400),
        max_depth=trial.suggest_int('max_depth', 5, 25),
        min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 10),
        random_state=42, n_jobs=-1)
    model.fit(X_tr, y_tr)
    return mean_absolute_error(y_val, model.predict(X_val))

rf_study = optuna.create_study(direction='minimize')
rf_study.optimize(rf_objective, n_trials=20)
print('Best RF params:', rf_study.best_params)

Best RF params: {'n_estimators': 276, 'max_depth': 17, 'min_samples_leaf': 10}


In [4]:
rf_best = RandomForestRegressor(**rf_study.best_params, random_state=42, n_jobs=-1)
rf_best.fit(X_train, y_train)
joblib.dump(rf_best, 'models/random_forest_optuna.pkl')

['models/random_forest_optuna.pkl']

### Tuning XGBoost

In [5]:
def xgb_objective(trial):
    model = XGBRegressor(
        n_estimators=trial.suggest_int('n_estimators', 100, 800),
        max_depth=trial.suggest_int('max_depth', 3, 10),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        subsample=trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
        random_state=42)
    model.fit(X_tr, y_tr)
    return mean_absolute_error(y_val, model.predict(X_val))

xgb_study = optuna.create_study(direction='minimize')
xgb_study.optimize(xgb_objective, n_trials=30)
print('Best XGB params:', xgb_study.best_params)

Best XGB params: {'n_estimators': 395, 'max_depth': 5, 'learning_rate': 0.014509896713239455, 'subsample': 0.7318648255478792, 'colsample_bytree': 0.6822119459420675}


In [6]:
xgb_best = XGBRegressor(**xgb_study.best_params, random_state=42)
xgb_best.fit(X_train, y_train)
joblib.dump(xgb_best, 'models/xgboost_optuna.pkl')

['models/xgboost_optuna.pkl']